# Step 4.3: size each problem in rupees per year, name an owner, rank

**The idea.** Steps 4.1 and 4.2 found many things. Now we put one rupee figure on each, for one fixed 12-month window, and say what *kind* of money it is.
Different kinds of money must not be added together. A claim that was paid is cash out. Stock sitting in a godown is a balance. Orders we could not fill are sales not made. So the table has a **money type** column and we rank only within the same type.

**The window.** The last 12 months of the data: FY26Q3 to FY27Q2, which is **1 Oct 2025 to 30 Sep 2026**. Claims are placed in the window by `claim_date`.
We do not divide a 2.5-year total by 2.5, because claims have been rising.

**What we do not do.** No margin, interest rate or cost of goods is in the data. So nothing here is a profit figure. Everything is at PTS (the price to the stockist) or at the claim value the client recorded. Where a figure would need an assumption, we say what is missing and leave it out.

**Owner.** The person at the client who runs the process the money passes through (from `notes/beliefs.md`). It is not blame; it is who to ask.

Run top to bottom. Each section ends with a **Check** line. Needs the outputs of `step4_rules.ipynb` in `data-clean/audit/rules/`.

In [1]:
import os, sys
import numpy as np, pandas as pd
sys.path.insert(0, os.path.join('..', 'pilot'))
from agent.data import load, fy_quarter
R = os.path.join('..', 'data-clean', 'audit', 'rules'); OUT = os.path.join('..', 'data-clean', 'audit', 'sizing'); os.makedirs(OUT, exist_ok=True)
ec = load('expiry_claims'); ps = load('primary_sales'); sku = load('sku_master').set_index('sku_code'); bat = load('batch_master')
W0, W1 = pd.Timestamp('2025-10-01'), pd.Timestamp('2026-09-30')
LQ = ['FY26Q3', 'FY26Q4', 'FY27Q1', 'FY27Q2']
inw = lambda s: (s >= W0) & (s <= W1)
ROWS = []
def row(id, problem, kind, inr, low, high, conf, owner, basis, source):
    ROWS.append(dict(id=id, problem=problem, money_type=kind, inr_12m=round(inr), low=round(low), high=round(high), confidence=conf, owner=owner, basis=basis, source=source))
    print(f'{id}: Rs {inr/1e7:.2f} cr  (range {low/1e7:.2f} to {high/1e7:.2f})  [{kind}]')
cr = lambda x: f'{x/1e7:.2f} crore'
print('window', W0.date(), 'to', W1.date())

window 2025-10-01 to 2026-09-30


## S1. Expiry, near-expiry, breakage and short-supply claims (the core problem)
**What it is.** Claims stockists lodged in the window. The client records each claim value as quantity times PTS (`01` p3 section 5.6), so this is what the company owes if every claim is paid.
**Money type.** Cash out (credit notes). Credited claims are already paid; pending claims are still owed.

In [2]:
c = ec[ec.status != 'Rejected']; cl = c[inw(c.claim_date)]
s1 = cl.claim_value_derived_inr.sum()
by_status = cl.groupby('status').claim_value_derived_inr.sum()
by_type = cl.groupby('claim_type').claim_value_derived_inr.sum()
print('claims lodged in window:', len(cl)); print((by_status/1e7).round(2).to_dict()); print((by_type/1e7).round(2).to_dict())
# trend by calendar quarter of claim_date: is it rising?
tr = c.groupby(c.claim_date.dt.to_period('Q')).claim_value_derived_inr.sum().div(1e7).round(2); print(tr.to_dict())
runrate = tr.iloc[-2:].sum() * 2 * 1e7                       # last two calendar quarters x 2
print('run-rate if the last 6 months continue:', cr(runrate))
# independent check: recompute value as qty x sku_master PTS, only where the recorded value is blank use it too
chk = (cl.qty_claimed * cl.sku_code.map(sku.pts_inr)).sum()
assert abs(chk - s1) < 1, (chk, s1)
row('S1', 'Expiry and related claims lodged', 'Cash out (claims)', s1, by_status.get('Credited', 0), runrate, 'High (recorded values = qty x PTS)',
    'Neha Bansal, Controller', 'claims lodged 1 Oct 2025 to 30 Sep 2026, non-rejected; low = credited only, high = last six months x 2', 'expiry_claims')
print('Check: total equals quantity x PTS recomputed from sku_master.')

claims lodged in window: 2294
{'Credited': 4.51, 'Pending': 0.96}
{'Breakage in transit': 0.04, 'Expired': 2.36, 'Near expiry': 3.04, 'Short supply': 0.03}
{Period('2024Q2', 'Q-DEC'): 0.01, Period('2024Q3', 'Q-DEC'): 0.33, Period('2024Q4', 'Q-DEC'): 0.31, Period('2025Q1', 'Q-DEC'): 0.55, Period('2025Q2', 'Q-DEC'): 0.63, Period('2025Q3', 'Q-DEC'): 0.86, Period('2025Q4', 'Q-DEC'): 1.21, Period('2026Q1', 'Q-DEC'): 1.15, Period('2026Q2', 'Q-DEC'): 1.41, Period('2026Q3', 'Q-DEC'): 1.69}
run-rate if the last 6 months continue: 6.20 crore
S1: Rs 5.47 cr  (range 4.51 to 6.20)  [Cash out (claims)]
Check: total equals quantity x PTS recomputed from sku_master.


## S2. Claims paid or pending that break a written rule (R1, R3, R4)
**What it is.** Claims in the window that fail a rule in `02`: outside the claim window (R1), more than was supplied (R3), or a probable duplicate (R4).
We count each claim once even if it breaks two rules. **Low** = only the claims where the breach is clear (R1 where the claim number and date agree). **High** = also the 81 R1 claims where the answer depends on which date is right.
**Caveat.** The data has no field for an approved exception, so some of these may be allowed.

In [3]:
r1 = pd.read_csv(os.path.join(R, 'r1_breaches.csv')); r3 = pd.read_csv(os.path.join(R, 'r3_breaches.csv')); r4 = pd.read_csv(os.path.join(R, 'r4_repeat_claims.csv'))
live = lambda d: d[d.status != 'Rejected']
clear = set(live(r1[r1.r1_confidence == 'clear']).claim_no) | set(live(r3).claim_no) | set(live(r4[r4.r4_probable_dup]).claim_no)
allr  = clear | set(live(r1).claim_no)
v = lambda ids: cl[cl.claim_no.isin(ids)].claim_value_derived_inr.sum()
lo, hi = v(clear), v(allr)
print('clear claims in window:', cl.claim_no.isin(clear).sum(), cr(lo), '| with date-dependent:', cl.claim_no.isin(allr).sum(), cr(hi))
row('S2', 'Claims that break a written rule (R1, R3, R4)', 'Cash out outside the rule', (lo + hi) / 2, lo, hi, 'Medium (no exception field in data)',
    'Neha Bansal, Controller', 'in-window claims failing R1/R3/R4, each claim once; midpoint of clear and date-dependent', 'step4_rules R1, R3, R4')
print('Check: sets are claims not rejected, counted once.')

clear claims in window: 261 0.74 crore | with date-dependent: 336 0.90 crore
S2: Rs 0.82 cr  (range 0.74 to 0.90)  [Cash out outside the rule]
Check: sets are claims not rejected, counted once.


## S3. Credit notes with no invoice reference and no evidenced approval (R2)
**What it is.** The rule says a claim without the original invoice needs written Area Manager approval. No remark mentions one. This is **exposure, not a proven loss**: the approval may exist on paper.

In [4]:
r2 = pd.read_csv(os.path.join(R, 'r2_no_invoice_no_approval.csv'))
x = cl[cl.claim_no.isin(set(r2.claim_no))]; s3 = x.claim_value_derived_inr.sum()
row('S3', 'Claims with no invoice ref and no approval shown (R2)', 'Exposure (not a loss)', s3, 0, s3, 'Low (approval may exist on paper)',
    'Neha Bansal, Controller', f'{len(x)} in-window claims; low = 0 because a valid approval would make it allowed', 'step4_rules R2')
print('Check: overlaps S1 and S2, so it is shown but never added to them.')

S3: Rs 0.98 cr  (range 0.00 to 0.98)  [Exposure (not a loss)]
Check: overlaps S1 and S2, so it is shown but never added to them.


## S4. Free goods above the 3% cap (R8)
**What it is.** Free goods given in the window, valued at PTS, and the part above the 3% of primary cap. Each quarter's excess was approved by the CFO (`06` p2), so this is the **size of an approved exception**, not a breach.

In [5]:
r8 = pd.read_csv(os.path.join(R, 'r8_free_goods_by_quarter.csv')); r8 = r8[r8.qtr.isin(LQ)]
free_tot, over = r8.free_goods_inr.sum(), r8.over_cap_inr.sum()
print('free goods in window:', cr(free_tot), 'of primary', cr(r8.primary_inr.sum()), f'= {free_tot/r8.primary_inr.sum()*100:.1f}%', '| above the cap:', cr(over))
row('S4', 'Free goods above the 3% cap (CFO-approved)', 'Approved cost (at PTS)', over, 0, free_tot, 'High (figures), but cost at PTS overstates the true cost',
    'Rohan Mehta, CFO', 'excess over 3% of primary, FY26Q3 to FY27Q2; high = all free goods at PTS', 'step4_rules R8')
print('Check: four quarters used, FY27Q2 is under the cap so adds zero.')

free goods in window: 6.63 crore of primary 201.42 crore = 3.3% | above the cap: 0.68 crore
S4: Rs 0.68 cr  (range 0.00 to 6.63)  [Approved cost (at PTS)]
Check: four quarters used, FY27Q2 is under the cap so adds zero.


## S5. Orders not filled (lost sales)
**What it is.** Order lines with no batch (not supplied) or fewer packs than ordered (short), valued at PTS. Short supply is allowed by the rule, so this is **sales not made**, not a breach. Some may have been filled later (back orders are not in the data), so this is an **upper bound**.

In [6]:
u = pd.read_csv(os.path.join(R, 'r11_unfilled_order_lines.csv')); u = u[u.qtr.isin(LQ)]
s5 = u.unfilled_inr.sum(); ns = u[u.status == 'not supplied'].unfilled_inr.sum()
print('unfilled value in window:', cr(s5), '| of which not supplied at all:', cr(ns)); print((u.groupby('qtr').unfilled_inr.sum()/1e7).round(2).to_dict())
row('S5', 'Order lines not filled in full (lost sales, at PTS)', 'Sales not made (revenue, not profit)', s5, ns, s5, 'Medium (back orders unknown)',
    'Kavita Rao, Head SC', 'unfilled value at PTS, FY26Q3 to FY27Q2; low = lines with no supply at all', 'step4_rules R11')
print('Check: this is revenue at PTS. Profit needs a margin, which the data does not have.')

unfilled value in window: 33.31 crore | of which not supplied at all: 31.98 crore
{'FY26Q3': 7.07, 'FY26Q4': 7.94, 'FY27Q1': 5.99, 'FY27Q2': 12.31}
S5: Rs 33.31 cr  (range 31.98 to 33.31)  [Sales not made (revenue, not profit)]
Check: this is revenue at PTS. Profit needs a margin, which the data does not have.


## S6. Stock above the 2-month norm (R10)
**What it is.** At the last snapshot (30 Sep 2026), packs above 2 months of average outward, at PTS. This is a **balance** (money tied up), not a yearly flow. Turning it into a yearly cost needs an interest rate that is not in the data.

In [7]:
r10 = pd.read_csv(os.path.join(R, 'r10_cover_by_cnf_sku_date.csv')); r10['snapshot_date'] = pd.to_datetime(r10.snapshot_date)
last = r10[r10.snapshot_date == r10.snapshot_date.max()]; s6 = last.excess_inr.sum()
print('snapshot', last.snapshot_date.iloc[0].date(), 'excess stock:', cr(s6)); print((last.groupby('city').excess_inr.sum()/1e7).round(2).to_dict())
row('S6', 'Stock above 2 months of cover at 30 Sep 2026', 'Balance (stock tied up)', s6, s6 * 0.86, s6, 'Medium (831 conflicting snapshot rows)',
    'Vaishali Kapoor, Demand Planning', 'balance, not a yearly cost; low = using the larger of conflicting rows (R10: 4.3 vs 5.0 crore)', 'step4_rules R10')
print('Check: this is a balance. It is not added to the yearly figures.')

snapshot 2026-09-30 excess stock: 4.96 crore
{'Bhiwandi': 0.72, 'Chennai': 0.83, 'Gurugram': 0.26, 'Hyderabad': 0.74, 'Kolkata': 2.41, 'Lucknow': 0.0}
S6: Rs 4.96 cr  (range 4.27 to 4.96)  [Balance (stock tied up)]
Check: this is a balance. It is not added to the yearly figures.


## S7. Where the claims come from: scheme batches (link from B3)
**What it is.** The share of in-window claim value that traces to batches invoiced on a scheme. It tells us **how much of S1 a change in scheme timing could reach**. It is not extra money.

In [8]:
sup = ps[ps.batch_no.notna()].copy(); sup['sch'] = sup.scheme_code.fillna('NA') != 'NO_SCHEME'
sb = sup.groupby(['stockist_code', 'batch_no']).sch.max().reset_index()
t = cl.merge(sb, on=['stockist_code', 'batch_no'], how='left'); tt = t[t.sch.notna()]
share = tt[tt.sch == True].claim_value_derived_inr.sum() / tt.claim_value_derived_inr.sum()
reach = share * s1
print(f'traceable in window: {len(tt)} of {len(t)} claims; scheme-batch share of traceable value = {share:.0%}; applied to S1 = {cr(reach)}')
row('S7', 'Part of S1 that traces to scheme-invoiced batches', 'Share of S1 (not extra money)', reach, reach * 0.8, reach, 'Medium (traced for most claims; goes with, not caused by)',
    'Arjun Sethi, NSM', 'scheme share of traceable in-window claim value x S1', 'step4_beliefs B3')
print('Check: shown for sizing the lever; excluded from the ranking total.')

traceable in window: 2294 of 2294 claims; scheme-batch share of traceable value = 91%; applied to S1 = 4.96 crore
S7: Rs 4.96 cr  (range 3.97 to 4.96)  [Share of S1 (not extra money)]
Check: shown for sizing the lever; excluded from the ranking total.


## Summary and ranking
Rank **within money type**. The one list that is comparable is cash out: S1 contains S2 and S3 overlaps both, so we show S1 as the size of the problem and S2 as the part that breaks a rule.

In [9]:
t = pd.DataFrame(ROWS)
t['crore'] = (t.inr_12m / 1e7).round(2); t['range_crore'] = (t.low / 1e7).round(2).astype(str) + ' to ' + (t.high / 1e7).round(2).astype(str)
order = {'Cash out (claims)': 1, 'Cash out outside the rule': 2, 'Sales not made (revenue, not profit)': 3, 'Approved cost (at PTS)': 4, 'Balance (stock tied up)': 5, 'Exposure (not a loss)': 6, 'Share of S1 (not extra money)': 7}
t['rank'] = t.money_type.map(order); t = t.sort_values(['rank', 'inr_12m'], ascending=[True, False]).drop(columns='rank')
t.to_csv(os.path.join(OUT, 'sizing_summary.csv'), index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 60)
print(t[['id', 'problem', 'money_type', 'crore', 'range_crore', 'confidence', 'owner']].to_string(index=False))
# the claims share of primary, for scale
prim = r8.primary_inr.sum(); print(f'\nScale: primary sales in the window {cr(prim)}; S1 is {s1/prim*100:.1f}% of it.')
assert len(t) == 7 and t.inr_12m.gt(0).all()
assert t.loc[t.id == 'S2', 'inr_12m'].iloc[0] < t.loc[t.id == 'S1', 'inr_12m'].iloc[0]
assert (t.low <= t.inr_12m + 1).all() and (t.inr_12m <= t.high + 1).all()
print('ALL SIZING CHECKS PASSED; wrote', os.path.join(OUT, 'sizing_summary.csv'))

id                                               problem                           money_type  crore    range_crore                                                confidence                            owner
S1                      Expiry and related claims lodged                    Cash out (claims)   5.47    4.51 to 6.2                        High (recorded values = qty x PTS)          Neha Bansal, Controller
S2         Claims that break a written rule (R1, R3, R4)            Cash out outside the rule   0.82    0.74 to 0.9                       Medium (no exception field in data)          Neha Bansal, Controller
S5   Order lines not filled in full (lost sales, at PTS) Sales not made (revenue, not profit)  33.31 31.98 to 33.31                              Medium (back orders unknown)              Kavita Rao, Head SC
S4            Free goods above the 3% cap (CFO-approved)               Approved cost (at PTS)   0.68    0.0 to 6.63  High (figures), but cost at PTS overstates the true cos